# 8 · Recommender: content, collaborative filtering and MMR

Discover serves a hybrid recommender (`folioverse_ml.recsys.RecModel`): candidates from semantic (LSA),
TF-IDF, theme and collaborative (NMF) retrieval, a linear score whose per-factor contributions are the
explanation, then maximal marginal relevance (MMR) with creator / franchise / media caps.

Offline protocol (`folioverse_ml.evaluate`): each synthetic user's earlier interactions are the profile and
their most recent ones are held out. This notebook reproduces the served evaluation and adds:

* a leave-one-factor-out ablation of the hybrid score;
* an MMR λ sweep (accuracy vs intra-list diversity);
* a cold-start breakdown by profile size.

All users and interactions are synthetic (DEMO).

In [1]:
import os
import sys

sys.path.insert(0, os.environ.get("FV_NOTEBOOK_DIR", os.getcwd()))
import fvnb

print("smoke mode" if fvnb.SMOKE else "full mode")

full mode


In [2]:
from collections import defaultdict

import numpy as np
from folioverse_ml.data import load_catalog, load_holdout
from folioverse_ml.evaluate import evaluate
from folioverse_ml.recsys import DEFAULT_WEIGHTS, FACTORS, Profile, RecModel, Request

items = load_catalog(fvnb.DATA / "synthetic/catalog/catalog.jsonl")
holdout = load_holdout(fvnb.DATA / "evaluation/rec-holdout")
with fvnb.Timer() as t:
    model = RecModel.train(items, holdout.train, seed=0)
print(
    f"{len(items)} items, {len(holdout.train)} train / {len(holdout.test)} test interactions, trained in {t.seconds:.1f}s"
)
print(model.manifest["training"])

K = 10
MAX_USERS = fvnb.sample(None, 40)

530 items, 5454 train / 1363 test interactions, trained in 0.7s
{'items': 530, 'users': 300, 'interactions': 5454, 'cfReconstructionErr': 57.1608}


## Served offline evaluation

In [3]:
report = evaluate(model, holdout, k=K, max_users=MAX_USERS)
cols = [
    "name",
    "precision",
    "recall",
    "ndcg",
    "hitRate",
    "coverage",
    "diversity",
    "novelty",
    "popularityPercentile",
]
fvnb.table(report["variants"], cols)
print(f"{report['users']} users, {report['durationMs']} ms")

| name | precision | recall | ndcg | hitRate | coverage | diversity | novelty | popularityPercentile |
|---|---|---|---|---|---|---|---|---|
| popularity | 0.019 | 0.043 | 0.032 | 0.180 | 0.030 | 0.888 | 3.531 | 0.990 |
| item_knn | 0.038 | 0.089 | 0.068 | 0.330 | 0.847 | 0.836 | 4.460 | 0.684 |
| nmf | 0.034 | 0.074 | 0.052 | 0.297 | 0.723 | 0.843 | 4.367 | 0.730 |
| content | 0.061 | 0.146 | 0.104 | 0.487 | 0.942 | 0.676 | 4.805 | 0.535 |
| hybrid | 0.059 | 0.140 | 0.103 | 0.447 | 0.898 | 0.702 | 4.611 | 0.618 |
| hybrid_mmr | 0.050 | 0.116 | 0.089 | 0.400 | 0.913 | 0.761 | 4.627 | 0.615 |

300 users, 8082 ms


## Notebook harness for ablations
Same split and scoring as `evaluate()`, with the request varied per experiment.

In [4]:
train: dict[str, dict[int, float]] = defaultdict(dict)
test: dict[str, set[int]] = defaultdict(set)
for it in holdout.train:
    if (i := model.index.get(it.item)) is not None:
        train[it.user][i] = max(train[it.user].get(i, 0.0), it.weight)
for it in holdout.test:
    if (i := model.index.get(it.item)) is not None:
        test[it.user].add(i)
users = sorted(u for u in test if train.get(u))[:MAX_USERS]
emb = model.features.embeddings


def run(req: Request, subset: list[str] | None = None) -> dict:
    nd, hit, ild = [], [], []
    for u in subset or users:
        profile = Profile(positives={model.items[i].id: w for i, w in train[u].items()})
        ranked = [model.index[r["id"]] for r in model.recommend(profile, req)["items"]]
        nd.append(fvnb.ndcg(ranked, dict.fromkeys(test[u], 1.0), K))
        hit.append(float(bool(set(ranked[:K]) & test[u])))
        if len(ranked) > 1:
            e = emb[ranked]
            m = len(ranked)
            ild.append(float((m * m - (e @ e.T).sum()) / (m * (m - 1))))
    return {"ndcg": fvnb.mean(nd), "hitRate": fvnb.mean(hit), "diversity": fvnb.mean(ild)}


def plain(weights=None, lam=1.0) -> Request:
    mmr = lam < 1.0
    return Request(
        k=K,
        weights=weights,
        mmr_lambda=lam,
        per_creator=2 if mmr else K,
        per_franchise=2 if mmr else K,
        balance_media=mmr,
    )

## Leave-one-factor-out

In [5]:
full = run(plain())
ablation = [{"variant": "all factors", **full}]
for f in FACTORS:
    w = {**DEFAULT_WEIGHTS, f: 0.0}
    r = run(plain(w))
    ablation.append({"variant": f"without {f}", **r, "delta_ndcg": r["ndcg"] - full["ndcg"]})
fvnb.table(ablation)

| variant | ndcg | hitRate | diversity |
|---|---|---|---|
| all factors | 0.103 | 0.447 | 0.702 |
| without semantic | 0.098 | 0.453 | 0.746 |
| without tfidf | 0.102 | 0.460 | 0.710 |
| without theme | 0.089 | 0.413 | 0.721 |
| without preference | 0.106 | 0.477 | 0.676 |
| without prior | 0.101 | 0.430 | 0.699 |

## MMR λ sweep

In [6]:
lambdas = [1.0, 0.9, 0.7, 0.5, 0.3] if not fvnb.SMOKE else [1.0, 0.7, 0.3]
mmr_rows = [{"lambda": lam, **run(plain(lam=lam))} for lam in lambdas]
fvnb.table(mmr_rows)

| lambda | ndcg | hitRate | diversity |
|---|---|---|---|
| 1.000 | 0.103 | 0.447 | 0.702 |
| 0.900 | 0.093 | 0.430 | 0.725 |
| 0.700 | 0.089 | 0.400 | 0.761 |
| 0.500 | 0.085 | 0.400 | 0.824 |
| 0.300 | 0.060 | 0.267 | 0.908 |

In [7]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 2.8))
ax.plot([r["diversity"] for r in mmr_rows], [r["ndcg"] for r in mmr_rows], marker="o")
for r in mmr_rows:
    ax.annotate(
        f"λ={r['lambda']}", (r["diversity"], r["ndcg"]), fontsize=8, xytext=(4, 4), textcoords="offset points"
    )
ax.set_xlabel("intra-list diversity")
ax.set_ylabel(f"nDCG@{K}")
fig.tight_layout()

## Cold start: profile size

In [8]:
sizes = np.array([len(train[u]) for u in users])
cuts = np.quantile(sizes, [1 / 3, 2 / 3])
buckets = {
    f"small (<={int(cuts[0])})": [u for u in users if len(train[u]) <= cuts[0]],
    "medium": [u for u in users if cuts[0] < len(train[u]) <= cuts[1]],
    f"large (>{int(cuts[1])})": [u for u in users if len(train[u]) > cuts[1]],
}
cold = []
for name, us in buckets.items():
    if us:
        cold.append(
            {
                "profile": name,
                "users": len(us),
                **{f"hybrid_{k}": v for k, v in run(plain(lam=0.7), us).items() if k != "diversity"},
            }
        )
fvnb.table(cold)

| profile | users | hybrid_ndcg | hybrid_hitRate |
|---|---|---|---|
| small (<=14) | 105 | 0.090 | 0.352 |
| medium | 108 | 0.095 | 0.398 |
| large (>22) | 87 | 0.082 | 0.460 |

## Findings

In [9]:
by = {v["name"]: v for v in report["variants"]}
fvnb.finding(
    f"Served hybrid+MMR nDCG@{K} {by['hybrid_mmr']['ndcg']:.3f} vs popularity {by['popularity']['ndcg']:.3f}, NMF {by['nmf']['ndcg']:.3f}."
)
fvnb.finding(
    f"MMR (λ=0.7) changes nDCG by {by['hybrid_mmr']['ndcg'] - by['hybrid']['ndcg']:+.3f} and diversity by "
    f"{by['hybrid_mmr']['diversity'] - by['hybrid']['diversity']:+.3f}; coverage {by['hybrid']['coverage']:.2f} -> {by['hybrid_mmr']['coverage']:.2f}."
)
worst = min(ablation[1:], key=lambda r: r["delta_ndcg"])
fvnb.finding(
    f"Most important factor: {worst['variant'].removeprefix('without ')} (removing it changes nDCG by {worst['delta_ndcg']:+.3f})."
)
fvnb.finding(
    f"Popularity bias (0.5 = none): popularity {by['popularity']['popularityPercentile']:.2f}, served {by['hybrid_mmr']['popularityPercentile']:.2f}."
)

- Served hybrid+MMR nDCG@10 0.089 vs popularity 0.032, NMF 0.052.
- MMR (λ=0.7) changes nDCG by -0.014 and diversity by +0.060; coverage 0.90 -> 0.91.
- Most important factor: theme (removing it changes nDCG by -0.014).
- Popularity bias (0.5 = none): popularity 0.99, served 0.62.


In [10]:
fvnb.save(
    "08-recsys",
    {
        "k": K,
        "users": len(users),
        "modelTraining": model.manifest["training"],
        "variants": report["variants"],
        "factorAblation": ablation,
        "mmrSweep": mmr_rows,
        "coldStart": cold,
    },
)

saved datasets/evaluation/results/nb-08-recsys.json
